### RAG pipeline - data ingestion to vector db

In [ ]:
# import required libraries
import os
from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader, DirectoryLoader
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
HF_TOKEN = os.getenv('HUGGING_FACE_API_KEY')

/tmp/ipykernel_258272/852626585.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader, DirectoryLoader


In [13]:
# Read all the pdf inside diractory
def load_all_pdf(path_to_pdfs):
    """Load and process all the pdfs"""
    all_documents = []
    path_dir = Path(path_to_pdfs)

    # load all pdf as list
    files = list(path_dir.glob('**/*.pdf'))

    if not files:
        return None
    print(f"Found {len(files)} pdfs.")
    
    for file in files:
        print(f"prpcessing {file}")
        try:
            loader = PyMuPDFLoader(str(file))
            documents = loader.load()
            print(f'found {len(documents)} pages\n')

            for doc in documents:
                doc.metadata['source_file'] = file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
        except Exception as e:
            print(f"Error: {e}")
    
    return all_documents


# less control over the files. Instead return all documents at once
def load_all_pdf2(dir):
    loader = DirectoryLoader(
        dir,
        glob='**/*.pdf',
        loader_cls=PyMuPDFLoader
    )
    all_documents = loader.load()

    for doc in all_documents:
        doc.metadata['source_file'] = Path(doc.metadata['file_path']).name
        doc.metadata['file_type'] = 'pdf'
    
    return all_documents

documents = load_all_pdf('../data/pdf/AstroPhysics/')
# documents

Found 4 pdfs.
prpcessing ../data/pdf/AstroPhysics/ImpactofPerfectFluidDarkMatter.pdf
found 33 pages

prpcessing ../data/pdf/AstroPhysics/PropagatingInstabilityforWaveDarkMatter.pdf
found 33 pages

prpcessing ../data/pdf/AstroPhysics/ParticlePhysicsandModern.pdf
found 24 pages

prpcessing ../data/pdf/AstroPhysics/Core-HaloRelationofQuantumWaveDarkMatter.pdf
found 6 pages



In [20]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Split into chunks 
def split_text(docs, chunk_size = 1000, chunk_overlap = 200):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators=['\n\n', '\n', '. ', ' ', '']
    )

    # Token-based chunking
    # splitter1 = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    #     chunk_size=512,
    #     chunk_overlap=64,
    #     separators=['\n\n', '\n', '. ', ' ', '']
    # )
    chunk = splitter.split_documents(documents=docs)
    print(f'Splitted {len(docs)} documents into {len(chunk)} chunk')

    if chunk:
        # print(f'{chunk[0].metadata}')
        print(f'{chunk[0].page_content[:200]}')

    return chunk

chunks = split_text(documents)

Splitted 96 documents into 382 chunk
Impact of Perfect Fluid Dark Matter on the
Appearance of Rotating Black Hole
Yu-Xiang Huanga, Sen Guob,∗, En-Wei Lianga,∗, Kai Linc
aGuangxi Key Laboratory for Relativistic Astrophysics, School of Phy


In [4]:
# Embedding models
models = [
    'sentence-transformers/all-MiniLM-L6-v2',
    'BAAI/bge-base-en-v1.5',    
    "BAAI/bge-small-en-v1.5",
    "intfloat/e5-base-v2"
]

In [5]:
## Embedding and vector store
from langchain_huggingface import HuggingFaceEndpointEmbeddings
from langchain_chroma import Chroma


hf_embeddings = HuggingFaceEndpointEmbeddings(  
    model="sentence-transformers/all-mpnet-base-v2",  
    task="feature-extraction",  
    huggingfacehub_api_token=HF_TOKEN,  
)

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=hf_embeddings,
    collection_name='chromaDb',
    persist_directory='../chroma_db'
)

print("Vector store created successfully!")

/home/desktop-potato/Desktop/ai/prj/rag-app/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Vector store created successfully!


In [6]:
from typing import List

class RAGRetriever:
    """Handles query based retrival from vector-store"""

    def __init__(self, vector_store = vector_store, embeddings = hf_embeddings):
        """
        Initialize Retriever
        ARG:
            vector_store: vector store containing the embeddings
            embeddings: embedding function to handle query embedding
        """

        self.vector_store = vector_store
        self.embeddings = embeddings
    
    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[dict[str, any]]:
        """
        Retrieve relevant document for a querry 

        arg:
            query: The search query 
            top_k: Number of top best result to return 
            score_threshold: Minimum similarity score
        Returns:
            List of relevant document and metadata
        """

        print(f'Retrieving relevant data for querry "{query}"')
        print(f'Top_k: {top_k}, score_threshold: {score_threshold}')

        # Generating query embeddings
        query_embedding = self.embeddings.embed_query(query)

        # Search in vector db
        try:
            results = vector_store._collection.query(
                query_embeddings=query_embedding,
                n_results=top_k
            )

            # process result
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                doc_ids = results['ids'][0]
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(doc_ids, documents, metadatas, distances)):
                    # Convert distance into similarity score
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'doc_id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f'Retrive {len(retrieved_docs)} documents after filtering.')
            else:
                print('No document found')
            
            return retrieved_docs
                

        except Exception as e:
            print(f'Exception occured with {e}')

retriever = RAGRetriever()

In [7]:
#  An example how to use the retriever
result = retriever.retrieve("Dark matter")

Retrieving relevant data for querry "Dark matter"
Top_k: 5, score_threshold: 0.0
Retrive 5 documents after filtering.


In [8]:
groq_model = "llama-3.1-8b-instant"
GROQ_API_KEY = os.getenv('GROQ_API_KEY')

In [9]:
from langchain_groq import ChatGroq
from langchain_classic.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

# Initialize the groq llm
llm = ChatGroq(
    api_key=GROQ_API_KEY,
    model=groq_model,
    temperature=0.1,
    # max_token = 1024
)

class GroqLLM: 
    def __init__(self, model_name: str, api_key: str, temperature: float = 0.1, max_token: int = 1024):
        """
        Initialize Groq LLM
        
        Args:
            model_name: Groq model name (qwen2-72b-instruct, llama3-70b-8192, etc.)
            api_key: Groq API key (or set GROQ_API_KEY environment variable)
        """

        if api_key is None:
            raise ValueError('groq api key required')
        self.model_name = model_name
        self.api_key = api_key

        # Initialize the groq llm
        self.llm = ChatGroq(
            api_key=GROQ_API_KEY,
            model=groq_model,
            temperature=temperature,
            # max_token = max_token
        )
    
        print("Groq llm initialized")
    
    def generate_response(self, query: str, context: str, max_length: int = 500) :
        """
        Generate response using the retrieved context
        
        Args:
            query: User question
            context: Retrieved document context
            max_length: Maximum response length
            
        Returns:
            Generated response string
        """

        # Create prompt template
        prompt_template = PromptTemplate(
            input_variables=["context"],
            template="You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.\n Context: {context}\nAnswer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."
        )
        
        formatted_prompt = prompt_template(context=context, question=query)

        messages = [
            SystemMessage(content=formatted_prompt),
            HumanMessage(content=query)
        ]
        try:
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Exception : {e}"
        
# groqLLM  = GroqLLM(model_name="llama-3.1-8b-instant", api_key=GROQ_API_KEY)
llm.invoke('Dark matter. be concise').content

"**Dark Matter:**\n\n- **Definition:** Invisible, non-luminous matter that makes up approximately 27% of the universe's mass-energy density.\n- **Detection:** Observed through its gravitational effects on visible matter and the large-scale structure of the universe.\n- **Properties:** No known composition, interacts with normal matter only through gravity.\n- **Theories:** WIMPs (Weakly Interacting Massive Particles), axions, and sterile neutrinos are potential candidates."

In [10]:
# Simple rag function: Context ritrival + generating response
def rag_func(query: str, retriever: RAGRetriever, llm) :
    # Retrive the context from vector database
    results = retriever.retrieve(query=query, top_k=5, score_threshold=0.0)
    context = '\n\n'.join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question"
    
    # Generate answer using llm
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    answer = llm.invoke(prompt.format(context = context, query = query)).content
    return answer

generated_answer = rag_func(query='what is dark matter', retriever=retriever, llm=llm)
generated_answer

Retrieving relevant data for querry "what is dark matter"
Top_k: 5, score_threshold: 0.0
Retrive 5 documents after filtering.


'Dark matter is a type of matter that is thought to make up approximately 26% of the universe, interacting primarily through self-gravity. It is composed of particles that do not emit, absorb, or reflect light, making it invisible and detectable only through its gravitational effects on visible matter.'

In [30]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
class RAGPipeline:
    _MAX_MESSAGES = 20
    def __init__(self, llm, retriever):
        if llm is None or retriever is None:
            raise ValueError("Both llm and retriever is required")
        
        self.llm = llm
        self.retriever = retriever
        self.history = []

        print("Pipeline initialized.")
    
    def __summarize_history(self):
        prompt = [
            SystemMessage(
                    content="""
                    Summarize the conversation.
                    Keep:
                    - User goals
                    - Important facts
                    - Decisions made
                    - Open questions

                    Be concise.
                    """
            )
        ] + self.history

        summary = self.llm.invoke(prompt).content

        self.history = self.history[:2] + [
            SystemMessage(content=f"Conversation summary:\n{summary}")
        ] + self.history[-2:]
    
    def query(self, query: str, top_k: int = 5, min_score: float = 0.1, stream: bool = False):
        if len(self.history) > self._MAX_MESSAGES:
            self.__summarize_history() 
        # Retrive related context
        retrieved_context = self.retriever.retrieve(query=query, top_k=top_k, score_threshold=min_score)

        if not retrieved_context:
            answer = "No relevant content found"
            self.history.append(
                AIMessage(content="No relevant content found.")
            )
        else:
            context = "\n|n".join(doc['content'] for doc in retrieved_context)
            sources = [{
                'content': doc['content'],
                'source': doc['metadata']['source_file'], 
                'page': doc['metadata']['page'],
                'score': doc['similarity_score'] 
                } for doc in retrieved_context
            ]
            prompt_template = PromptTemplate(
                input_variables=["context", "query"],
                template="""
                Answer the user's QUESTION using the DOCUMENT text above.
                Keep your answer grounded in the facts of the DOCUMENT.
                If the DOCUMENT does not contain enough information, return NONE.

                Context:
                {context}

                Question:
                {query}
                """
            )
            prompt = prompt_template.format(context=context, query = query)
            self.history.append(HumanMessage(content=query))
            
            response = self.llm.invoke(prompt)
            self.history.append(AIMessage(content=response.content))
            answer = response.content

        return {
            "answer": answer,
            "sources": sources,
        }     

rag_pipeline = RAGPipeline(llm=llm, retriever=retriever)    

Pipeline initialized.


In [35]:
response = rag_pipeline.query('Define Dark matter') 
response['answer']

Retrieving relevant data for querry "Define Dark matter"
Top_k: 5, score_threshold: 0.1
Retrive 4 documents after filtering.


'Dark matter is a type of matter that interacts primarily through self-gravity and is thought to comprise approximately 26% of the universe.'

In [36]:
response['sources']

[{'content': 'olution, capable of resolving the smallest galaxy halos\nforming in this context [39].\nWarm dark matter (WDM) is also capable of suppress-\ning small-scale linear power by free streaming [40], but it\nsuﬀers from the Catch 22 problem [41], where the light\nparticle mass required for creating a suﬃciently large core\n(∼1 kpc) would prevent the formation of dwarf galaxies\nin the ﬁrst place. Collisional CDM does somewhat bet-\nter in producing cores consistent with observations, but\nit cannot suppress the number of dwarf galaxies [42, 43].\nFor these reasons, ψDM and scalar-ﬁeld dark matter\ncomposed of extremely light particles have recently be-\ngun to attract attention as a viable contender for the\nlong-sought dark matter (e.g., [39, 44–51]).\nCosmic structures at high redshifts provide stringent\ntests for all alternative dark matter models attempting to\nsolve the small-scale issues of CDM in the Local Group.\nFor WDM a tension arises when requiring the relatively',